In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

# **Preguntas a responder:**
**1. ¿Qué categoría genera más ingreso total?**  
**2. ¿Qué región tiene mejor margen de ganancia?**  
**3. ¿Qué subcategorías están destruyendo margen?**

## Importación y configuración del DataFrame

In [2]:
df = pd.read_csv("product_sales_dataset_final.csv")

In [16]:
# Quitamos espacios al inicio y al final de los nombres de cada columna
df.columns = df.columns.str.strip()

In [4]:
df_pBI = df.copy()

In [18]:
# Convertimos las fechas en formato MM-DD-YY
df_pBI["Order_Date"] = pd.to_datetime(
    df_pBI["Order_Date"], format="%m-%d-%y", errors="coerce"
)

In [6]:
# Columnas de apoyo temporal
df_pBI["Year"] = df_pBI["Order_Date"].dt.year
df_pBI["Month_Name"] = df_pBI["Order_Date"].dt.month_name()
df_pBI["Quarter"] = "Q" + df_pBI["Order_Date"].dt.quarter.astype(str)

In [19]:
# Limpieza basica de espacios en cadenas de texto
text_cols = [
    "Customer_Name",
    "City",
    "State",
    "Region",
    "Country",
    "Category",
    "Sub_Category",
    "Product_Name",
]
for col in text_cols:
    df_pBI[col] = df_pBI[col].astype(str).str.strip()

## Dimensiones del DataFrame

In [8]:
print(f"El DataFrame tiene {df_pBI.shape[0]:,.0f} filas")
print(f"El DataFrame tiene {df_pBI.shape[1]:,.0f} columnas")

El DataFrame tiene 200,000 filas
El DataFrame tiene 17 columnas


## Análisis exploratorio

In [9]:
df_pBI.isna().sum()

Order_ID         0
Order_Date       0
Customer_Name    0
City             0
State            0
Region           0
Country          0
Category         0
Sub_Category     0
Product_Name     0
Quantity         0
Unit_Price       0
Revenue          0
Profit           0
Year             0
Month_Name       0
Quarter          0
dtype: int64

In [10]:
df_pBI.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 17 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Order_ID       200000 non-null  int64         
 1   Order_Date     200000 non-null  datetime64[us]
 2   Customer_Name  200000 non-null  str           
 3   City           200000 non-null  str           
 4   State          200000 non-null  str           
 5   Region         200000 non-null  str           
 6   Country        200000 non-null  str           
 7   Category       200000 non-null  str           
 8   Sub_Category   200000 non-null  str           
 9   Product_Name   200000 non-null  str           
 10  Quantity       200000 non-null  int64         
 11  Unit_Price     200000 non-null  float64       
 12  Revenue        200000 non-null  float64       
 13  Profit         200000 non-null  float64       
 14  Year           200000 non-null  int32         
 15  Month_Name 

In [11]:
df

,Order_ID,Order_Date,Customer_Name,City,State,Region,Country,Category,Sub_Category,Product_Name,Quantity,Unit_Price,Revenue,Profit
0,1,08-23-23,Bianca Brown,Jackson,Mississippi,South,United States,Accessories,Small Electronics,Phone Case,3,201.01,603.03,221.49
1,2,12-20-24,Jared Edwards,Grand Rapids,Michigan,Centre,United States,Accessories,Small Electronics,Charging Cable,4,74.30,297.20,97.09
2,3,01-29-24,Susan Valdez,Minneapolis,Minnesota,Centre,United States,Clothing & Apparel,Sportswear,Nike Air Force 1,1,68.19,68.19,25.47
3,4,11-29-24,Tina Williams,Tallahassee,Florida,South,United States,Clothing & Apparel,Sportswear,Adidas Tracksuit,3,209.64,628.92,231.38
4,5,09-21-23,Catherine Gordon,Baltimore,Maryland,East,United States,Accessories,Bags,Backpack,1,216.63,216.63,42.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,199996,08-15-23,William Jackson,Boston,Massachusetts,East,United States,Home & Furniture,Storage,Storage Rack,4,254.42,1017.68,334.72
199996,199997,10-17-23,Sharon Ferrell,Bismarck,North Dakota,Centre,United States,Accessories,Small Electronics,Charging Cable,1,237.04,237.04,46.53
199997,199998,12-03-23,Katie Rivera,Santa Fe,New Mexico,West,United States,Accessories,Wearable Accessories,Sunglasses,2,106.83,213.66,102.57
199998,199999,12-08-23,Lisa Sullivan,New York City,New York,East,United States,Clothing & Apparel,Women's Wear,Zara Blouse,2,353.01,706.02,288.74


**1. ¿Qué categoría genera más ingreso total?** La categoría que genera mas **ventas totales** es la de _"Electronics"_ con un monto total: $\$57,485,698.06\text{ USD}$

In [12]:
# Agrupar por Categoría y sumar Revenue
ingreso_categoria = df_pBI.groupby("Category")["Revenue"].sum().sort_values(ascending=False).reset_index()
ingreso_categoria

,Category,Revenue
0,Electronics,57485698.06
1,Home & Furniture,47674426.96
2,Clothing & Apparel,27134365.30
3,Accessories,10113254.61


**2. ¿Qué región tiene mejor margen de ganancia?** La región con mejor margen de ganancia es **_"South"_** (Sur), ya que la misma genera un margen de ganancia de **23.57%**

In [13]:
# Creamos la columna de margen % por fila
df_pBI["Profit_Margin_%"] = (df_pBI["Profit"] / df_pBI["Revenue"]) * 100

# Agrupamos sumando los totales de Profit y Revenue por Region
margen_region = (
    df_pBI.groupby("Region")[["Profit", "Revenue"]].sum().reset_index()
)

# Calculamos el Margen de Ganancia real agrupado (%)
margen_region["Profit_Margin_%"] = (
    margen_region["Profit"] / margen_region["Revenue"]
) * 100

# Ordenamos de mayor a menor
margen_region = margen_region.sort_values(
    by="Profit_Margin_%", ascending=False
).reset_index(drop=True)

margen_region

,Region,Profit,Revenue,Profit_Margin_%
0,South,5918454.17,25102960.64,23.576718
1,West,8313962.76,36242841.73,22.939600
2,Centre,8094863.77,36081894.34,22.434697
3,East,9221327.43,44980048.22,20.500928


**3. ¿Qué subcategorías están destruyendo margen?**  
Para identificar las subcategorías que están "destruyendo margen", debemos buscar aquellas cuyo margen de ganancia **(Profit_Margin_%)** sea el más bajo o incluso negativo comparado con el promedio global del negocio.  

Las subcategorías que están destruyendo margen (al tener los márgenes de ganancia más bajos del negocio, situándose alrededor del **13.97% - 14.08%**) son:

1. **Laptops:** 13.97%
2. **Home Appliances:** 13.99%
3. **Wearables:** 14.03%
4. **Tablets:** 14.04%
5. **Smartphones:** 14.08%
6. **TVs & Audio:** 14.09%

In [14]:
# Agrupamos sumando Profit y Revenue por Subcategoría
margen_subcat = (
    df_pBI.groupby("Sub_Category")[["Profit", "Revenue"]].sum().reset_index()
)

# Calculamos el Margen de Ganancia real agrupado (%)
margen_subcat["Profit_Margin_%"] = (
    margen_subcat["Profit"] / margen_subcat["Revenue"]
) * 100

# Ordenamos de MENOR a MAYOR para ver las peores arriba
margen_subcat = margen_subcat.sort_values(
    by="Profit_Margin_%", ascending=True
).reset_index(drop=True)

margen_subcat.head(10)

,Sub_Category,Profit,Revenue,Profit_Margin_%
0,Laptops,1726382.83,12358319.81,13.969398
1,Home Appliances,1208198.78,8638544.88,13.986138
2,Wearables,1293256.90,9216507.48,14.031963
3,Tablets,1175512.12,8373830.76,14.037925
4,Smartphones,1535723.81,10904335.31,14.083608
5,TVs & Audio,1126039.48,7994159.82,14.085776
6,Home Decor,1895514.27,8070216.82,23.487774
7,Furniture,2281148.34,9697778.92,23.522379
8,Bedding,3068113.04,13042783.30,23.523453
9,Storage,1823643.82,7746306.75,23.542107


## Exportación del DataFrame

In [20]:
# Exportar DataFrame limpio a CSV, para luego hacer el dashboard en PowerBI
df_pBI.to_csv("ventas_procesadas_pBI.csv", index=False, encoding="utf-8-sig")

print("Archivo exportado con exito")

Archivo exportado con exito
